#  Preprocessing Travel Tourism Dataset

This notebook prepares the data for machine learning models (Logistic Regression, SVMs, Neural Networks).  
We will perform:

- Removing rows with missing target
- Handling missing values
- Dropping columns with excessive emptiness
- Categorical encoding (One-hot, label mappings)
- Converting Yes/No columns to binary
- Saving processed datasets for model notebooks


### Import Libraries

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# For saving processed files
import os


### Load Datasets and show shapes

In [2]:
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

train.head()

Train Shape: (12654, 25)
Test Shape: (5852, 24)


,trip_id,country,age_group,travel_companions,num_females,num_males,main_activity,visit_purpose,is_first_visit,mainland_stay_nights,...,food_included,domestic_transport_included,sightseeing_included,guide_included,insurance_included,days_booked_before_trip,arrival_weather,total_trip_days,has_special_requirements,spend_category
0,tour_idftaa27vp,FRANCE,45-64,With Spouse and Children,1.0,2.0,Beach Tourism,Leisure and Holidays,Yes,0,...,No,No,No,No,No,NaN,"cloudy,",30+,NaN,1.0
1,tour_iduck75m57,KENYA,45-64,Alone,1.0,0.0,Conference Tourism,Meetings and Conference,Yes,6,...,No,No,No,No,No,15-30,"sunny,",30+,NaN,2.0
2,tour_id8y3w40h8,SOUTH AFRICA,25-44,With Other Friends/Relatives,2.0,0.0,Cultural Tourism,Meetings and Conference,No,4,...,No,No,No,No,No,90+,"sunny,",30+,none,2.0
3,tour_idkoh8mkgr,ITALY,25-44,With Spouse,1.0,1.0,Widlife Tourism,Leisure and Holidays,Yes,0,...,Yes,Yes,Yes,Yes,No,8-14,NaN,NaN,none,0.0
4,tour_idkmsfa00a,ITALY,25-44,With Spouse,1.0,1.0,Beach Tourism,Leisure and Holidays,Yes,0,...,Yes,No,No,No,No,90+,"sunny,",7-14,NaN,0.0


### Drop non-learning identifiers and seperate target

In [3]:
# Save trip_id for submission
train_trip_id = train["trip_id"]
test_trip_id = test["trip_id"]

# Remove trip_id from feature set
train = train.drop(columns=["trip_id"])
test = test.drop(columns=["trip_id"])

# Extract target
target = train["spend_category"]

print("Target distribution:")
print(target.value_counts(normalize=True).round(3))

Target distribution:
spend_category
0.0    0.495
1.0    0.389
2.0    0.116
Name: proportion, dtype: float64


### Drop rows with missing target

In [4]:
# Count before removal
initial_rows = train.shape[0]

# Remove missing labels
clean_train = train.dropna(subset=["spend_category"]).copy()
clean_target = clean_train["spend_category"]

rows_after = clean_train.shape[0]

print(f"Removed {initial_rows - rows_after} rows with missing target.")
print("New train shape:", clean_train.shape)

Removed 34 rows with missing target.
New train shape: (12620, 24)


### Train Validation Split

In [5]:
X = clean_train.drop(columns=["spend_category"])
y = clean_target

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train set:", X_train.shape)
print("Validation set:", X_val.shape)

Train set: (10096, 23)
Validation set: (2524, 23)


### Remove columns with >90% missing values

In [7]:
missing_percent = X_train.isnull().mean() * 100
cols_to_drop = missing_percent[missing_percent > 90].index.tolist()

print("Columns dropped (>90% missing):")
print(cols_to_drop)

X_train.drop(columns=cols_to_drop, inplace=True)
X_val.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=cols_to_drop, inplace=True)

Columns dropped (>90% missing):
[]


### Handling missing Numerical values

In [10]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    median_val = X_train[col].median()

    X_train[col] = X_train[col].fillna(median_val)
    X_val[col]  = X_val[col].fillna(median_val)
    test[col]   = test[col].fillna(median_val)

print("Numerical missing values handled.")

Numerical missing values handled.


### Handling missing Categorical Values

In [11]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    X_train[col] = X_train[col].fillna("Unknown")
    X_val[col]  = X_val[col].fillna("Unknown")
    test[col]   = test[col].fillna("Unknown")

print("Categorical missing values handled.")

Categorical missing values handled.


### Convert Yes/No Columns to Binary

In [12]:

binary_cols = [
    "is_first_visit",
    "intl_transport_included",
    "accomodation_included",
    "food_included",
    "domestic_transport_included",
    "sightseeing_included",
    "guide_included",
    "insurance_included"
]

yes_no_map = {"Yes": 1, "No": 0}

for col in binary_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].map(yes_no_map).fillna(0)
        X_val[col]   = X_val[col].map(yes_no_map).fillna(0)
        test[col]    = test[col].map(yes_no_map).fillna(0)

print("Converted Yes/No columns to binary.")


Converted Yes/No columns to binary.


### Encoding Categorical features

In [18]:
# Identify categorical columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

# Split columns based on cardinality
low_cardinality_cols = [c for c in cat_cols if X_train[c].nunique() <= 20]
high_cardinality_cols = [c for c in cat_cols if X_train[c].nunique() > 20]

print("Low-cardinality OHE columns:\n", low_cardinality_cols)
print("\nHigh-cardinality Ordinal columns:\n", high_cardinality_cols)

# ----- OHE for low cardinality -----
ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
ohe.fit(X_train[low_cardinality_cols])

X_train_ohe = ohe.transform(X_train[low_cardinality_cols])
X_val_ohe   = ohe.transform(X_val[low_cardinality_cols])
test_ohe    = ohe.transform(test[low_cardinality_cols])

# ----- Ordinal encoding for high cardinality -----
ord_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
ord_enc.fit(X_train[high_cardinality_cols])

X_train_ord = ord_enc.transform(X_train[high_cardinality_cols])
X_val_ord   = ord_enc.transform(X_val[high_cardinality_cols])
test_ord    = ord_enc.transform(test[high_cardinality_cols])


# Drop original categorical cols
X_train_num = X_train.drop(columns=cat_cols)
X_val_num   = X_val.drop(columns=cat_cols)
test_num    = test.drop(columns=cat_cols)

# Convert OHE to DataFrames
ohe_cols = ohe.get_feature_names_out(low_cardinality_cols)

X_train_final = pd.concat([
    X_train_num.reset_index(drop=True),
    pd.DataFrame(X_train_ohe, columns=ohe_cols),
    pd.DataFrame(X_train_ord, columns=high_cardinality_cols)
], axis=1)

X_val_final = pd.concat([
    X_val_num.reset_index(drop=True),
    pd.DataFrame(X_val_ohe, columns=ohe_cols),
    pd.DataFrame(X_val_ord, columns=high_cardinality_cols)
], axis=1)

test_final = pd.concat([
    test_num.reset_index(drop=True),
    pd.DataFrame(test_ohe, columns=ohe_cols),
    pd.DataFrame(test_ord, columns=high_cardinality_cols)
], axis=1)

print("\nEncoding complete")
print("Train final shape:", X_train_final.shape)
print("Val final shape:", X_val_final.shape)
print("Test final shape:", test_final.shape)


Low-cardinality OHE columns:
 ['age_group', 'travel_companions', 'main_activity', 'visit_purpose', 'tour_type', 'info_source', 'days_booked_before_trip', 'arrival_weather', 'total_trip_days', 'has_special_requirements']

High-cardinality Ordinal columns:
 ['country']

Encoding complete
Train final shape: (10096, 79)
Val final shape: (2524, 79)
Test final shape: (5852, 79)


### Save Processed Data

In [20]:
os.makedirs("../data/processed", exist_ok=True)

X_train_final.to_csv("../data/processed/X_train.csv", index=False)
X_val_final.to_csv("../data/processed/X_val.csv", index=False)
test_final.to_csv("../data/processed/test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)

# Save submission IDs
test_trip_id.to_csv("../data/processed/test_trip_id.csv", index=False)

print("Saved processed datasets successfully")

Saved processed datasets successfully
